In [ ]:
!date

### process pycistopic

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import scanpy as sc
import anndata

import pycisTopic
from pycisTopic.lda_models import evaluate_models

from pycisTopic.clust_vis import (
    find_clusters,
    run_umap,
    run_tsne,
    plot_metadata,
    plot_topic,
    cell_topic_heatmap
)
from pycisTopic.topic_binarization import binarize_topics
from pycisTopic.topic_qc import compute_topic_metrics, plot_topic_qc, topic_annotation
import matplotlib.pyplot as plt
from pycisTopic.utils import fig2img
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
import numpy as np
from pycisTopic.utils import region_names_to_coordinates
import os

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))

In [ ]:
indir = f'{projdir}/atac/pycistopic/obj'
n = 30
with open(f'{indir}/', 'rb') as fin:
    model = pickle.load(fin)

In [ ]:
indir = f'{projdir}/atac/pycistopic/models'
n = 30
with open(f'{indir}/{n}_topics.model.intersect.pkl', 'rb') as fin:
    model = pickle.load(fin)

In [ ]:
### I think any more than 100 topics and it starts to overfit?
indir = f'{projdir}/atac/pycistopic/models'

all_models = []
for n in range(10, 300, 10):
#     print(n)
    try:
        with open(f'{indir}/{n}_topics.model.intersect.pkl', 'rb') as fin:
            model = pickle.load(fin)
        all_models.append(model[0])
    except:
        print(f'{n} not found')
        continue

In [ ]:
model = evaluate_models(
    all_models,
#     select_model = 60,
    return_model = True
)

In [ ]:
indir = f'{projdir}/atac/pycistopic/models'

all_models = []
for n in range(5, 100, 5):
    print(n)
    try:
        with open(f'{indir}/{n}_topics.model.intersect.pkl', 'rb') as fin:
            model = pickle.load(fin)
        all_models.append(model[0])
    except:
        print(f'{n} not found')
        continue

In [ ]:
model = evaluate_models(
    all_models,
#     select_model = 60,
    return_model = True
)

### hard to say... but let's choose 75 topics for now
- too many and it doesn't feel great...?
- https://github.com/aertslab/pycisTopic/issues/150

In [ ]:
indir = f'{projdir}/atac/pycistopic/models'
n = 75
with open(f'{indir}/{n}_topics.model.intersect.pkl', 'rb') as fin:
    final_model = pickle.load(fin)

In [ ]:
indir = f'{projdir}/atac/pycistopic/obj'
with open(f'{indir}/merged.intersect.pkl', 'rb') as fin:
    cistopic_obj = pickle.load(fin)

In [ ]:
cistopic_obj.add_LDA_model(final_model[0])

In [ ]:
cistopic_obj

In [ ]:
clusters = pd.read_csv(f'{projdir}/csv/clusters/rna_leiden_v6.csv',
                       sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
cistopic_obj.add_cell_data(clusters[['coarse_v6']])
cistopic_obj.add_cell_data(clusters[['clusters_v6']])

In [ ]:
find_clusters(
    cistopic_obj,
    target  = 'cell',
    k = 10,
    res = [0.6, 1.2, 3],
    prefix = 'pycisTopic_',
    scale = True,
    split_pattern = '-'
)

In [ ]:
cistopic_obj.projections['cell'] = {}

In [ ]:
run_umap(cistopic_obj, scale=True, random_state=0)

In [ ]:
umap = cistopic_obj.projections['cell']['UMAP']
umap.to_csv(f'{projdir}/csv/atac/pycistopic/intersect_rna.{n}.umap.csv', sep='\t', header=True, index=True)

In [ ]:
plot_metadata(
    cistopic_obj,
    reduction_name='UMAP',
    variables=['clusters_v6', 'coarse_v6'],
    target='cell', num_columns=4,
    text_size=10,
    dot_size=5)

### let's check topic-based clustering on RNA UMAP

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac_reindex_small.h5ad')

In [ ]:
adata

In [ ]:
wnn = pd.read_csv(f'{projdir}/csv/meta/joint/wnn_umap.csv', sep='\t', header=0, index_col=0)
adata.obsm['X_umap'] = wnn.values

In [ ]:
adata.obs['coarse_v6'] = clusters.reindex(adata.obs.index)['coarse_v6']
adata.obs['clusters_v6'] = clusters.reindex(adata.obs.index)['clusters_v6']

In [ ]:
sc.pl.umap(adata, color=['time_num'])

In [ ]:
adata.obs['pycistopic_leiden'] = cistopic_obj.cell_data.reindex(adata.obs.index)['pycisTopic_leiden_10_1.2']

In [ ]:
### ok it's actually not bad lol
sc.pl.umap(adata, color=['coarse_v6', 'pycistopic_leiden'])

In [ ]:
sc.pl.umap(adata, color=['clusters_v6', 'pycistopic_leiden'], legend_loc='on data')

In [ ]:
annot_dict = {}
for resolution in [0.6, 1.2, 3]:
    annot_dict[f"pycisTopic_leiden_10_{resolution}"] = {}
    for cluster in set(cistopic_obj.cell_data[f"pycisTopic_leiden_10_{resolution}"]):
        counts = cistopic_obj.cell_data.loc[
            cistopic_obj.cell_data.loc[cistopic_obj.cell_data[f"pycisTopic_leiden_10_{resolution}"] == cluster].index,
            "coarse_v6"].value_counts()
        annot_dict[f"pycisTopic_leiden_10_{resolution}"][cluster] = f"{counts.index[counts.argmax()]}({cluster})"

In [ ]:
# annot_dict

In [ ]:
plot_topic(
    cistopic_obj,
    reduction_name = 'UMAP',
    target = 'cell',
    num_columns=5
)

In [ ]:
cell_topic_heatmap(
    cistopic_obj,
    variables = ['coarse_v6'],
    scale = False,
    legend_loc_x = 1.0,
    legend_loc_y = -1.2,
    legend_dist_y = -1,
    figsize = (10, 10)
)

In [ ]:
region_bin_topics_top_3k = binarize_topics(
    cistopic_obj, method='ntop', ntop = 3_000,
    plot=True, num_columns=5
)

In [ ]:
region_bin_topics_otsu = binarize_topics(
    cistopic_obj, method='otsu',
    plot=True, num_columns=5
)

In [ ]:
binarized_cell_topic = binarize_topics(
    cistopic_obj,
    target='cell',
    method='li',
    plot=True,
    num_columns=5, nbins=100)

In [ ]:
topic_qc_metrics = compute_topic_metrics(cistopic_obj)

In [ ]:
fig_dict={}
fig_dict['CoherenceVSAssignments']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Log10_Assignments', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['AssignmentsVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Log10_Assignments', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSRegions_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Regions_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSMarginal_dist']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Marginal_topic_dist', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSGini_index']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Gini_index', var_color='Gini_index', plot=False, return_fig=True)

In [ ]:
topic_annot = topic_annotation(
    cistopic_obj,
    annot_var='clusters_v6',
    binarized_cell_topic=binarized_cell_topic,
    general_topic_thr = 0.2
)

In [ ]:
topic_annot

### I think we stop here and get a bigger notebook

In [ ]:
# with open(f'{outdir}/merged.intersect.{n}.preprocess.pkl', 'wb') as fout:
#     pickle.dump(fout)

In [ ]:
outdir = f'{projdir}/atac/pycistopic/final_model'
pickle.dump(cistopic_obj, open(f'{outdir}/merged.intersect.{n}.preprocess.pkl', 'wb')) 

In [ ]:
os.makedirs(os.path.join(outdir, "region_sets"), exist_ok = True)
os.makedirs(os.path.join(outdir, "region_sets", "Topics_otsu"), exist_ok = True)
os.makedirs(os.path.join(outdir, "region_sets", "Topics_top_3k"), exist_ok = True)
os.makedirs(os.path.join(outdir, "region_sets", "DARs_cell_type"), exist_ok = True)

In [ ]:
for topic in region_bin_topics_otsu:
    region_names_to_coordinates(
        region_bin_topics_otsu[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(outdir, "region_sets", "Topics_otsu", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
for topic in region_bin_topics_top_3k:
    region_names_to_coordinates(
        region_bin_topics_top_3k[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(outdir, "region_sets", "Topics_top_3k", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
# for cell_type in markers_dict:
#     region_names_to_coordinates(
#         markers_dict[cell_type].index
#     ).sort_values(
#         ["Chromosome", "Start", "End"]
#     ).to_csv(
#         os.path.join(out_dir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"),
#         sep = "\t",
#         header = False, index = False
#     )

In [ ]:
# imputed_acc_obj = impute_accessibility(
#     cistopic_obj,
#     selected_cells=None,
#     selected_regions=None,
#     scale_factor=10**6
# )

In [ ]:
# normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

In [ ]:
!date